# XchangeFinder — Interactive Data Analysis & Visualization

**Original Project**: SocGen Finhack 2017 — Team 17 (FinNoobs)  
**App**: Currency Exchange Shop Finder (Hong Kong focused)

This notebook provides **interactive** analysis with Plotly charts (hover, zoom, pan, 3D rotate), comparison tables, and architecture diagrams.


## 1. Setup & Libraries

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

DB_PATH = Path('../db/XchangeFinder.db')
conn = sqlite3.connect(DB_PATH)
print('Connected to:', DB_PATH.resolve())

def query(sql: str) -> pd.DataFrame:
    return pd.read_sql_query(sql, conn)

print('Libraries loaded (pandas + plotly).')


## 2. Database Schema Overview

In [ ]:
tables = query("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name;")
print('Tables in database:')
display(tables)

for t in tables['name']:
    print(f'\n=== {t} ===')
    info = query(f'PRAGMA table_info({t})')
    display(info[['name', 'type', 'notnull', 'pk']])


## 3. Load All Tables

In [ ]:
stores = query('SELECT * FROM stores')
users = query('SELECT * FROM users')
user_rates = query('SELECT * FROM userRateSubmissions')
store_rates = query('SELECT * FROM storeRateSubmissions')
safety = query('SELECT * FROM safetyRatings')

print(f'Stores          : {len(stores)} rows')
print(f'Users           : {len(users)} rows')
print(f'User Rates      : {len(user_rates)} rows')
print(f'Store Rates     : {len(store_rates)} rows')
print(f'Safety Ratings  : {len(safety)} rows')


## 4. Comparison Tables

### 4.1 Store Summary Comparison Table


In [ ]:
store_summary = stores[['storeUUID', 'displayName', 'latitude', 'longitude', 'time']].copy()
store_summary = store_summary.rename(columns={'displayName': 'Store', 'time': 'Opening Hours'})

avg_s = safety.groupby('storeUUID')['rating'].agg(['mean', 'count']).reset_index()
avg_s.columns = ['storeUUID', 'Avg Safety', 'Num Ratings']
store_summary = store_summary.merge(avg_s, on='storeUUID', how='left')

ur_cnt = user_rates.groupby('storeUUID').size().reset_index(name='User Rate Subs')
store_summary = store_summary.merge(ur_cnt, on='storeUUID', how='left')

sr_cnt = store_rates.groupby('storeUUID').size().reset_index(name='Official Rate Subs')
store_summary = store_summary.merge(sr_cnt, on='storeUUID', how='left')

store_summary = store_summary.fillna(0)
store_summary['Avg Safety'] = store_summary['Avg Safety'].round(2)
store_summary = store_summary.sort_values('Avg Safety', ascending=False).reset_index(drop=True)

print('=== STORE COMPARISON TABLE ===')
display(store_summary)


### 4.2 Currency Pair Rate Comparison (User vs Store)

In [ ]:
user_pair = user_rates.groupby(['fromCurrency', 'toCurrency']).agg(
    User_Avg_Rate=('rate', 'mean'),
    User_Min=('rate', 'min'),
    User_Max=('rate', 'max'),
    User_Count=('rate', 'count')
).reset_index()

store_pair = store_rates.groupby(['fromCurrency', 'toCurrency']).agg(
    Store_Avg_Rate=('rate', 'mean'),
    Store_Min=('rate', 'min'),
    Store_Max=('rate', 'max'),
    Store_Count=('rate', 'count')
).reset_index()

rate_comparison = pd.merge(user_pair, store_pair, on=['fromCurrency', 'toCurrency'], how='outer').fillna(0)
rate_comparison['Pair'] = rate_comparison['fromCurrency'] + ' → ' + rate_comparison['toCurrency']
rate_comparison['Difference (User - Store)'] = (rate_comparison['User_Avg_Rate'] - rate_comparison['Store_Avg_Rate']).round(4)
rate_comparison = rate_comparison[[
    'Pair', 'User_Avg_Rate', 'Store_Avg_Rate', 'Difference (User - Store)',
    'User_Count', 'Store_Count', 'User_Min', 'User_Max', 'Store_Min', 'Store_Max'
]].round(4)

print('=== USER vs STORE RATE COMPARISON ===')
display(rate_comparison.sort_values('User_Count', ascending=False))


### 4.3 Safety Rating Leaderboard

In [ ]:
safety_detail = safety.merge(stores[['storeUUID', 'displayName']], on='storeUUID', how='left')
safety_leaderboard = safety_detail.groupby('displayName').agg(
    Average_Rating=('rating', 'mean'),
    Total_Votes=('rating', 'count'),
    Min_Rating=('rating', 'min'),
    Max_Rating=('rating', 'max')
).round(2).sort_values('Average_Rating', ascending=False)

print('=== SAFETY LEADERBOARD ===')
display(safety_leaderboard)


## 5. Interactive 2D Charts (Plotly)

### 5.1 Average Safety Rating by Store

In [ ]:
fig = px.bar(
    store_summary, x='Store', y='Avg Safety',
    color='Avg Safety', color_continuous_scale='Viridis',
    text='Avg Safety',
    title='Average Safety Rating by Exchange Store',
    labels={'Avg Safety': 'Average Safety (1-5)'},
    hover_data=['Num Ratings', 'User Rate Subs', 'Official Rate Subs', 'Opening Hours']
)
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(yaxis_range=[0, 5.5], height=450)
fig.show()


### 5.2 Rate Submissions: User vs Store (Grouped Bar)

In [ ]:
sub_long = store_summary.melt(
    id_vars=['Store'],
    value_vars=['User Rate Subs', 'Official Rate Subs'],
    var_name='Source', value_name='Submissions'
)
sub_long['Source'] = sub_long['Source'].replace({
    'User Rate Subs': 'User Submissions',
    'Official Rate Subs': 'Store Official Rates'
})

fig = px.bar(
    sub_long, x='Store', y='Submissions', color='Source',
    barmode='group',
    title='Rate Submissions: Crowdsourced (User) vs Official (Store)',
    color_discrete_map={'User Submissions': '#3498db', 'Store Official Rates': '#e74c3c'},
    text='Submissions'
)
fig.update_traces(textposition='outside')
fig.update_layout(height=450)
fig.show()


### 5.3 Interactive Geographic Map of Stores

In [ ]:
# Align safety to stores order
map_df = stores.merge(
    store_summary[['storeUUID', 'Avg Safety', 'Num Ratings', 'User Rate Subs', 'Official Rate Subs']],
    on='storeUUID', how='left'
)

fig = px.scatter(
    map_df, x='longitude', y='latitude',
    size='Avg Safety', color='Avg Safety',
    hover_name='displayName',
    hover_data={
        'longitude': ':.5f', 'latitude': ':.5f',
        'Avg Safety': ':.2f', 'Num Ratings': True,
        'User Rate Subs': True, 'Official Rate Subs': True,
        'time': True
    },
    color_continuous_scale='RdYlGn', range_color=[1, 5],
    size_max=40,
    title='Hong Kong Exchange Stores — Geographic Locations (Size & Color = Safety)',
    labels={'longitude': 'Longitude', 'latitude': 'Latitude'}
)
fig.update_layout(height=550)
fig.show()


### 5.4 Interactive Heatmap — Currency Pair Rates (User)

In [ ]:
if len(user_rates) > 0:
    heat_data = user_rates.pivot_table(
        index='fromCurrency', columns='toCurrency', values='rate', aggfunc='mean'
    )
    fig = px.imshow(
        heat_data,
        text_auto='.3f',
        color_continuous_scale='YlOrRd',
        title='User-Submitted Exchange Rate Heatmap',
        labels=dict(x='To Currency', y='From Currency', color='Avg Rate'),
        aspect='auto'
    )
    fig.update_layout(height=500)
    fig.show()
else:
    print('No user rate data for heatmap.')


### 5.5 Safety Rating Distribution

In [ ]:
fig = px.histogram(
    safety, x='rating', nbins=5,
    title='Distribution of Safety Ratings',
    labels={'rating': 'Safety Rating (1-5)', 'count': 'Frequency'},
    color_discrete_sequence=['#9b59b6'],
    marginal='box'
)
fig.update_layout(bargap=0.15, height=450)
fig.update_xaxes(dtick=1)
fig.show()


### 5.6 Boxplot — Rate Distribution by Pair (User vs Store)

In [ ]:
user_c = user_rates.copy(); user_c['Source'] = 'User'
store_c = store_rates.copy(); store_c['Source'] = 'Store'
combined = pd.concat([
    user_c[['fromCurrency', 'toCurrency', 'rate', 'Source']],
    store_c[['fromCurrency', 'toCurrency', 'rate', 'Source']]
], ignore_index=True)
combined['Pair'] = combined['fromCurrency'] + '→' + combined['toCurrency']

fig = px.box(
    combined, x='Pair', y='rate', color='Source',
    title='Rate Distribution by Currency Pair — User vs Store',
    color_discrete_map={'User': '#3498db', 'Store': '#e74c3c'},
    points='all',
    hover_data=['fromCurrency', 'toCurrency']
)
fig.update_layout(height=500)
fig.show()


### 5.7 Top Active Users

In [ ]:
top_users = user_rates.groupby('username').size().reset_index(name='Submissions').sort_values('Submissions', ascending=False)

fig = px.bar(
    top_users, x='username', y='Submissions',
    color='Submissions', color_continuous_scale='Plasma',
    text='Submissions',
    title='Most Active Users (Rate Submissions)'
)
fig.update_traces(textposition='outside')
fig.update_layout(height=400)
fig.show()


### 5.8 Correlation Heatmap (Interactive)

In [ ]:
corr_df = store_summary[['Avg Safety', 'Num Ratings', 'User Rate Subs', 'Official Rate Subs']].corr().round(3)
print('Correlation Matrix:')
display(corr_df)

fig = px.imshow(
    corr_df, text_auto=True, color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Correlation between Safety & Activity Metrics',
    aspect='auto'
)
fig.update_layout(height=450)
fig.show()


## 6. Interactive 3D Charts (Plotly)

### 6.1 3D Scatter — Latitude × Longitude × Safety Rating\n\n*Drag to rotate, scroll to zoom*

In [ ]:
map3d = stores.merge(
    store_summary[['storeUUID', 'Avg Safety', 'Num Ratings', 'User Rate Subs']],
    on='storeUUID', how='left'
)

fig = px.scatter_3d(
    map3d, x='longitude', y='latitude', z='Avg Safety',
    color='Avg Safety', size='Avg Safety',
    hover_name='displayName',
    hover_data={
        'longitude': ':.5f', 'latitude': ':.5f',
        'Avg Safety': ':.2f', 'Num Ratings': True, 'User Rate Subs': True
    },
    color_continuous_scale='RdYlGn', range_color=[1, 5],
    size_max=25,
    title='3D View: Store Location + Safety Rating',
    labels={'longitude': 'Longitude', 'latitude': 'Latitude', 'Avg Safety': 'Avg Safety'}
)
fig.update_layout(
    height=650,
    scene=dict(
        zaxis=dict(range=[0, 5.5]),
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
    )
)
fig.show()


### 6.2 3D Bar-style view — Submissions by Store

In [ ]:
# Create 3D mesh-style bars using scatter3d + lines for visual height
fig = go.Figure()

stores_list = store_summary['Store'].tolist()
user_vals = store_summary['User Rate Subs'].tolist()
store_vals = store_summary['Official Rate Subs'].tolist()

for i, (name, u, s) in enumerate(zip(stores_list, user_vals, store_vals)):
    # User bar
    fig.add_trace(go.Scatter3d(
        x=[i, i], y=[0, 0], z=[0, u],
        mode='lines+markers',
        line=dict(color='#3498db', width=12),
        marker=dict(size=6, color='#3498db'),
        name=f'{name} — User',
        hovertemplate=f'{name}<br>User Submissions: {u}<extra></extra>',
        showlegend=True if i == 0 else False,
        legendgroup='user'
    ))
    # Store bar
    fig.add_trace(go.Scatter3d(
        x=[i, i], y=[1, 1], z=[0, s],
        mode='lines+markers',
        line=dict(color='#e74c3c', width=12),
        marker=dict(size=6, color='#e74c3c'),
        name=f'{name} — Store',
        hovertemplate=f'{name}<br>Store Submissions: {s}<extra></extra>',
        showlegend=True if i == 0 else False,
        legendgroup='store'
    ))

fig.update_layout(
    title='3D Submissions by Store (User vs Official)',
    height=650,
    scene=dict(
        xaxis=dict(tickvals=list(range(len(stores_list))), ticktext=stores_list, title='Store'),
        yaxis=dict(tickvals=[0, 1], ticktext=['User', 'Store'], title='Source'),
        zaxis=dict(title='Submissions'),
        camera=dict(eye=dict(x=1.6, y=1.6, z=1.1))
    )
)
fig.show()


### 6.3 3D Rate Comparison — User vs Store by Currency Pair

In [ ]:
if len(rate_comparison) > 0:
    rc = rate_comparison.copy()
    fig = go.Figure()

    pairs = rc['Pair'].tolist()
    x = list(range(len(pairs)))
    user_r = rc['User_Avg_Rate'].tolist()
    store_r = rc['Store_Avg_Rate'].tolist()

    fig.add_trace(go.Scatter3d(
        x=x, y=[0]*len(x), z=user_r,
        mode='lines+markers',
        name='User Avg Rate',
        line=dict(color='#3498db', width=5),
        marker=dict(size=8, color='#3498db'),
        hovertemplate='Pair: %{text}<br>User Rate: %{z:.4f}<extra></extra>',
        text=pairs
    ))
    fig.add_trace(go.Scatter3d(
        x=x, y=[1]*len(x), z=store_r,
        mode='lines+markers',
        name='Store Avg Rate',
        line=dict(color='#e74c3c', width=5),
        marker=dict(size=8, color='#e74c3c'),
        hovertemplate='Pair: %{text}<br>Store Rate: %{z:.4f}<extra></extra>',
        text=pairs
    ))
    # Connecting difference lines
    for i in range(len(x)):
        fig.add_trace(go.Scatter3d(
            x=[x[i], x[i]], y=[0, 1], z=[user_r[i], store_r[i]],
            mode='lines',
            line=dict(color='gray', width=2, dash='dash'),
            showlegend=False,
            hoverinfo='skip'
        ))

    fig.update_layout(
        title='3D Rate Comparison: User vs Store by Currency Pair',
        height=650,
        scene=dict(
            xaxis=dict(tickvals=x, ticktext=pairs, title='Currency Pair'),
            yaxis=dict(tickvals=[0, 1], ticktext=['User', 'Store'], title='Source'),
            zaxis=dict(title='Exchange Rate'),
            camera=dict(eye=dict(x=1.5, y=1.8, z=1.0))
        )
    )
    fig.show()


## 7. Interactive Architecture Overview

In [ ]:
# Simple interactive network-style view of components
nodes = [
    dict(id='Browser', label='Browser\n(HTML+Bootstrap+Raty)', group=1),
    dict(id='Flask', label='Flask Routes\n(views.py)', group=2),
    dict(id='Forms', label='Forms & Auth\n(forms/login)', group=2),
    dict(id='SQLite', label='SQLite DB', group=3),
    dict(id='Forex', label='forex-python', group=4),
    dict(id='Maps', label='Google Maps API', group=4),
]

# Use a simple scatter with annotations for architecture
fig = go.Figure()

# Positions
pos = {
    'Browser': (0, 2), 'Flask': (2, 2), 'Forms': (4, 2),
    'SQLite': (0, 0), 'Forex': (2, 0), 'Maps': (4, 0)
}
colors = {'Browser': '#5dade2', 'Flask': '#58d68d', 'Forms': '#f4d03f',
          'SQLite': '#ec7063', 'Forex': '#af7ac5', 'Maps': '#e67e22'}

for name, (x, y) in pos.items():
    fig.add_trace(go.Scatter(
        x=[x], y=[y], mode='markers+text',
        marker=dict(size=55, color=colors[name], line=dict(width=2, color='black')),
        text=[name], textposition='middle center',
        textfont=dict(size=11, color='white', family='Arial Black'),
        name=name,
        hovertemplate=f'<b>{name}</b><extra></extra>'
    ))

# Edges
edges = [('Browser','Flask'), ('Flask','Forms'), ('Flask','SQLite'),
         ('Flask','Forex'), ('Flask','Maps')]
for a, b in edges:
    fig.add_trace(go.Scatter(
        x=[pos[a][0], pos[b][0]], y=[pos[a][1], pos[b][1]],
        mode='lines', line=dict(color='#2c3e50', width=2),
        showlegend=False, hoverinfo='skip'
    ))

fig.update_layout(
    title='XchangeFinder — System Components (Interactive)',
    height=450, showlegend=False,
    xaxis=dict(visible=False, range=[-0.8, 4.8]),
    yaxis=dict(visible=False, range=[-0.8, 2.8]),
    plot_bgcolor='white'
)
fig.show()


## 8. Live Forex Rates (for comparison)

In [ ]:
try:
    from forex_python.converter import CurrencyRates
    c = CurrencyRates()
    live_pairs = [('USD','CNY'), ('USD','HKD'), ('EUR','USD'), ('HKD','CNY'), ('SGD','USD'), ('EUR','CNY')]
    live_rows = []
    for frm, to in live_pairs:
        try:
            rate = c.get_rate(frm, to)
            live_rows.append({'Pair': f'{frm} → {to}', 'Live Rate (forex-python)': round(rate, 4)})
        except Exception as e:
            live_rows.append({'Pair': f'{frm} → {to}', 'Live Rate (forex-python)': None})
    live_df = pd.DataFrame(live_rows)
    print('=== LIVE OFFICIAL RATES ===')
    display(live_df)

    # Interactive bar of live rates
    live_ok = live_df.dropna()
    if len(live_ok) > 0:
        fig = px.bar(live_ok, x='Pair', y='Live Rate (forex-python)',
                     color='Live Rate (forex-python)', color_continuous_scale='Teal',
                     text='Live Rate (forex-python)',
                     title='Live Official Exchange Rates (forex-python)')
        fig.update_traces(texttemplate='%{text:.4f}', textposition='outside')
        fig.update_layout(height=420)
        fig.show()
except ImportError:
    print('Install forex-python: pip install forex-python')
except Exception as e:
    print(f'Could not fetch live rates: {e}')


## 9. Key Insights & Summary

| Insight | Finding |
|---------|---------|
| Stores | 3 exchange shops (Naboo, Gift Shop, Endor) — Star Wars themed |
| Users | 5 registered users |
| Crowdsourcing | Both users and shops can submit exchange rates |
| Safety | Shops receive 1–5 star safety ratings |
| Geography | All stores in Hong Kong (lat ~22.3–22.4, lon ~114.1–114.2) |
| Goal | Help users find trustworthy rates & safe shops |

### Interactive features in this notebook
- **Hover** for details on every point/bar
- **Zoom / Pan** on 2D charts
- **Rotate** 3D charts by dragging
- **Legend click** to show/hide series


In [ ]:
print('Interactive analysis complete.')
conn.close()
